# COSC-651 Week 1: Agentic Workflows and Degrees of Autonomy

Supports course learning outcomes 1, 5, and 7.

A finance team reports: "Blank invoice dates are accepted with the wrong date." Compare three requests to the same AI model to see what changes when you improve the instructions and then add findings and company rules.

You are investigating advice from an AI support workflow that was rushed from prototype into production. A finance team reports: "Blank invoice dates are accepted with the wrong date." Keep that complaint unchanged while you improve the request and then supply findings and company rules. Run three versions and compare the two changes.

## Experimental question

What changes after improving the instructions, and what changes after adding diagnostic findings and company rules?

Read the weekly lab and rubric before changing the starter.

## Environment

Use the configured course environment and a clean kernel. Preview the inputs before enabling live access.

In [ ]:
import json
import sys
from pathlib import Path
REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / 'src/maintenance_course/data.py').is_file()), None)
assert REPO_ROOT is not None, 'Open this notebook from a COSC-651-Labs checkout.'
sys.path[:0] = [str(REPO_ROOT / 'src'), str(REPO_ROOT)]
from maintenance_course.data import LAB_ROOT, case_by_id
from maintenance_course.experiments import save_runs
from maintenance_course.tools import ToolRegistry

print({"lab_root": str(LAB_ROOT), "live_requests_enabled": False})

## Run the three prompt versions

Read **Week 1 Lab | Compare Prompt Design and Evidence** in Canvas. Run D01 with the minimal prompt, the improved six-task prompt, and that same improved prompt with diagnostic findings and current rules. Save all three replies, then compare the two changes.

These are plain-text support replies. Preparing the prompts makes no model-service requests.

Choose one completion route below: browser chat or the configured CLI/API adapter. Both produce the same three-version comparison. Browser chat uses your existing model subscription; it needs no API key or separate API purchase.

The file and folder names below are defaults for the supplied helpers. Your submission can keep the same evidence in notebook cells, separate files, or a combination, under names you choose.

In [ ]:
import hashlib
import os

from maintenance_course.prompt_comparison import (
    build_three_prompts,
    import_browser_comparison,
    prepare_browser_comparison,
    run_prompt_comparison,
)

PROMPT_RUN_ROUTE = "cli"  # Choose "browser" or "cli" before your actual runs.
if PROMPT_RUN_ROUTE not in {"browser", "cli"}:
    raise ValueError('Choose PROMPT_RUN_ROUTE = "browser" or "cli".')

prompt_case = case_by_id("D01")
prompt_registry = ToolRegistry(prompt_case)
prompt_diagnostic = prompt_registry.call(
    "diagnose_csv", {"fixture_id": prompt_case["fixture_id"]}
)
prompt_policy_result = prompt_registry.call("read_document", {"doc_id": "dates-v2"})
assert prompt_policy_result["ok"], prompt_policy_result
prompt_policy = prompt_policy_result["result"]
three_prompts = build_three_prompts(prompt_case, prompt_diagnostic, prompt_policy)
for version, text in three_prompts.items():
    print(version)
    print(text)
    print()

def prompt_configuration():
    return {
        name: os.environ.get(name)
        for name in (
            "COURSE_BACKEND", "COURSE_MODEL", "COURSE_EFFORT", "GEMINI_MODEL"
        )
    }

def prompt_input_hashes():
    return {
        version: hashlib.sha256(text.encode("utf-8")).hexdigest()
        for version, text in three_prompts.items()
    }

def prompt_source_hash():
    sources = {
        "case": prompt_case, "diagnostic": prompt_diagnostic, "policy": prompt_policy
    }
    return hashlib.sha256(
        json.dumps(sources, sort_keys=True, ensure_ascii=False).encode("utf-8")
    ).hexdigest()

def check_prompt_preview():
    current_prompts = build_three_prompts(
        prompt_case, prompt_diagnostic, prompt_policy
    )
    if current_prompts != three_prompts:
        raise ValueError(
            "Prompts changed; rerun the input preview and start a separate comparison."
        )

def check_record_prompt_hashes(records):
    planned_hashes = prompt_input_hashes()
    for record in records:
        actual_hash = hashlib.sha256(record["prompt"].encode("utf-8")).hexdigest()
        if (
            record["prompt_hash"] != planned_hashes[record["variant"]]
            or actual_hash != record["prompt_hash"]
        ):
            raise ValueError(
                "A request used changed prompts. Its saved record is preserved; "
                "start a separate comparison."
            )

def load_cli_prompt_records(path):
    records = json.loads(path.read_text(encoding="utf-8"))
    variants = [r["variant"] for r in records]
    if variants not in (
        ["version1"], ["version1", "version2"],
        ["version1", "version2", "version3"]
    ):
        raise ValueError("Preserve this file; it contains a different comparison.")
    pilot = records[0]
    for record in records:
        if record["mode"] != "live" or record["case_id"] != prompt_case["id"]:
            raise ValueError("Saved records must belong to this live D01 comparison.")
        if record.get("comparison_configuration") != prompt_configuration():
            raise ValueError("Model settings changed; start a separate comparison.")
        if record.get("comparison_prompt_hashes") != prompt_input_hashes() or (
            record.get("comparison_inputs_hash") != prompt_source_hash()
        ):
            raise ValueError("Comparison inputs changed; start a separate comparison.")
        if record["model"] != pilot["model"] or record["settings"] != pilot["settings"]:
            raise ValueError("Saved model metadata changed; preserve this comparison.")
    check_prompt_preview()
    check_record_prompt_hashes(records)
    return records


### Browser route: export the three prompts

Set `PROMPT_RUN_ROUTE` to `"browser"` in the input cell and run that cell again. Then enable `PREPARE_BROWSER_FILES` below once. It creates a new capture folder and prints its location. It never replaces an existing folder.

Open `version1-prompt.txt`, send its exact contents in a fresh Claude, ChatGPT or Grok chat, and save the complete reply as `version1-response.txt` in that folder. Inspect the reply before repeating this for Versions 2 and 3, each in a new chat. Keep the same model and visible settings. Save browser errors too.

Fill in `capture.json`: provider, displayed model, visible settings and each request's date. Use JSON `null` for a model label or setting the interface does not show. Record the fresh-chat and unchanged-prompt attestations honestly. These are your reported conditions; the importer cannot verify the provider's hidden state. Keep `inputs.json` unchanged.

This route makes three manual model requests and no automated model calls. Leave both CLI run flags disabled.

In [ ]:
PREPARE_BROWSER_FILES = False
BROWSER_CAPTURE_FOLDER = LAB_ROOT / "artifacts/week-01-browser-capture"

if PREPARE_BROWSER_FILES:
    if PROMPT_RUN_ROUTE != "browser":
        raise ValueError('Select the "browser" route before exporting prompts.')
    check_prompt_preview()
    prepare_browser_comparison(
        prompt_case, prompt_diagnostic, prompt_policy, BROWSER_CAPTURE_FOLDER
    )
    print("Capture folder:", BROWSER_CAPTURE_FOLDER)
    print("Keep its prompts and inputs.json unchanged. Fill in capture.json.")
else:
    print("Browser export disabled. Enable it once if using browser chat.")


### Browser route: import and inspect your saved replies

After saving all three replies and completing `capture.json`, enable `IMPORT_BROWSER_REPLIES` once. The cell checks the prompt files and source inputs, imports your replies verbatim and saves three live records with manual-import provenance. It retains errors and leaves unavailable system instructions, token counts and elapsed time unknown. It will not overwrite an existing comparison.

Set both browser flags back to `False` afterward. With the browser route selected, a clean run of the notebook reads and displays the saved import without repeating requests or overwriting replies. Keep the exact transferred messages and their source inputs available for inspection, in notebook cells or separate files.

In [ ]:
IMPORT_BROWSER_REPLIES = False
prompt_browser_path = LAB_ROOT / "artifacts/week-01-prompt-comparison-browser.json"

if IMPORT_BROWSER_REPLIES:
    if PROMPT_RUN_ROUTE != "browser":
        raise ValueError('Select the "browser" route before importing replies.')
    if prompt_browser_path.exists():
        raise FileExistsError(
            f"Preserve {prompt_browser_path}; choose a new filename for another comparison."
        )
    check_prompt_preview()
    browser_prompt_records = import_browser_comparison(
        prompt_case, prompt_diagnostic, prompt_policy, BROWSER_CAPTURE_FOLDER
    )
    for record in browser_prompt_records:
        record["comparison_prompt_hashes"] = prompt_input_hashes()
        record["comparison_inputs_hash"] = prompt_source_hash()
    save_runs(prompt_browser_path, browser_prompt_records)

if PROMPT_RUN_ROUTE == "browser" and prompt_browser_path.is_file():
    browser_prompt_records = json.loads(prompt_browser_path.read_text(encoding="utf-8"))
    if [r["variant"] for r in browser_prompt_records] != [
        "version1", "version2", "version3"
    ] or any(
        r["mode"] != "live" or r.get("transport") != "manual_browser_import"
        or r["case_id"] != prompt_case["id"]
        or r.get("comparison_inputs_hash") != prompt_source_hash()
        for r in browser_prompt_records
    ):
        raise ValueError("Preserve this browser file; its comparison inputs differ.")
    check_prompt_preview()
    check_record_prompt_hashes(browser_prompt_records)
    for record in browser_prompt_records:
        print(record["variant"], record["provider"], record["model"], record["error"])
        print(record["response_text"] or record.get("browser_error_text"))
        print()
    print("Saved browser comparison:", prompt_browser_path)
else:
    print("No browser comparison loaded. Import saved replies if using this route.")


### CLI/API route: run the live Version 1 pilot

Skip these two live-run cells if you chose browser chat. Keep `PROMPT_RUN_ROUTE` set to `"cli"` for this route. Configure the course-approved backend and model using the setup guide before enabling this cell. For the Claude CLI route, use the authenticated local CLI session and the assigned `COURSE_BACKEND`, `COURSE_MODEL`, and `COURSE_EFFORT` settings. Do not put credentials in the notebook.

Set `RUN_PROMPT_PILOT` to `True` to make one live request. Inspect the actual reply, model, settings, usage, and error before running the other two versions. The file is saved immediately, including a failed request. An existing live file is preserved; select a new filename for a separate or repaired comparison.

The full prompt comparison uses three logical requests, including this pilot. Keep any repeated or repaired comparison separate.

In [ ]:
RUN_PROMPT_PILOT = False
prompt_live_path = LAB_ROOT / "artifacts/week-01-prompt-comparison-live.json"

if RUN_PROMPT_PILOT:
    if PROMPT_RUN_ROUTE != "cli":
        raise ValueError('Select the "cli" route before running this cell.')
    if prompt_live_path.exists():
        raise FileExistsError(
            f"Preserve {prompt_live_path}; choose a new filename for a new comparison."
        )
    check_prompt_preview()
    pilot_configuration = prompt_configuration()
    pilot_records = run_prompt_comparison(
        prompt_case, prompt_diagnostic, prompt_policy,
        mode="live", variants=("version1",)
    )
    for record in pilot_records:
        record["comparison_configuration"] = pilot_configuration
        record["comparison_prompt_hashes"] = prompt_input_hashes()
        record["comparison_inputs_hash"] = prompt_source_hash()
    save_runs(prompt_live_path, pilot_records)
    check_record_prompt_hashes(pilot_records)
    print(json.dumps(pilot_records[0], indent=2))
else:
    print("Live pilot disabled. Configure access and inspect the prompts first.")


### CLI/API route: keep the pilot and run Versions 2 and 3

After inspecting a successful pilot, set `RUN_REMAINING_PROMPTS` to `True`. This cell reuses the saved Version 1 record and makes only two additional requests. Each version uses a fresh model session with the same configuration. Each reply or request error is saved before the next request starts. If execution stops after Version 2 is saved, this cell resumes with Version 3. It never repeats a saved Version 2, including a failed one. The exact prompts and responses stay in the saved records. Leave the flag off when inspecting the starter without making requests.

In [ ]:
RUN_REMAINING_PROMPTS = False

if RUN_REMAINING_PROMPTS:
    if PROMPT_RUN_ROUTE != "cli":
        raise ValueError('Select the "cli" route before running this cell.')
    if not prompt_live_path.is_file():
        raise FileNotFoundError("Run and inspect the Version 1 pilot first.")
    prompt_live_records = load_cli_prompt_records(prompt_live_path)
    if [r["variant"] for r in prompt_live_records] not in (
        ["version1"], ["version1", "version2"]
    ):
        raise ValueError("This file already contains a completed or different comparison.")
    pilot = prompt_live_records[0]
    if pilot["mode"] != "live" or pilot["case_id"] != prompt_case["id"]:
        raise ValueError("The pilot must be the live D01 request for this comparison.")
    if pilot["error"] is not None:
        raise ValueError("Preserve the failed pilot and resolve access before continuing.")
    if pilot["prompt"] != three_prompts["version1"]:
        raise ValueError("The prompt changed; start a separately named comparison.")
    for version in ("version1", "version2", "version3")[len(prompt_live_records):]:
        record = run_prompt_comparison(
            prompt_case, prompt_diagnostic, prompt_policy,
            mode="live", variants=(version,)
        )[0]
        record["comparison_configuration"] = prompt_configuration()
        record["comparison_prompt_hashes"] = prompt_input_hashes()
        record["comparison_inputs_hash"] = prompt_source_hash()
        prompt_live_records.append(record)
        save_runs(prompt_live_path, prompt_live_records)
        check_record_prompt_hashes(prompt_live_records)
        if record["model"] != pilot["model"] or record["settings"] != pilot["settings"]:
            raise ValueError(
                "Model metadata changed. The response was saved; "
                "retain the records and start a separate comparison."
            )
    assert len(prompt_live_records) == 3
    for record in prompt_live_records:
        print(record["variant"], record["model"], record["error"])
        print(record["response_text"])
        print()
    print("Saved all three replies:", prompt_live_path)
elif PROMPT_RUN_ROUTE == "cli" and prompt_live_path.is_file():
    prompt_live_records = load_cli_prompt_records(prompt_live_path)
    for record in prompt_live_records:
        print(record["variant"], record["model"], record["settings"], record["error"])
        print(record["response_text"])
        print()
    print(f"Loaded {len(prompt_live_records)} of 3 saved requests:", prompt_live_path)
else:
    print("Remaining live versions disabled. Retain the pilot before enabling them.")


### Compare the two changes

Write two short notes in your notebook or a file of your choice:

- **Version 1 to Version 2:** what changed after improving the instructions?
- **Version 2 to Version 3:** what changed after adding findings and rules?

Quote the response text supporting each conclusion. Check for unsupported claims, missing information, and advice that follows or conflicts with the rules. An unchanged or worse result is still a result to explain.

Retain your three actual request-response records from the browser import or CLI/API run, including exact prompts, metadata, failures, and provenance. If course access requires a designated recorded comparison, preserve that status. After saving all three replies, set all four run/export/import flags back to `False` before restarting and running all cells. Existing live files are preserved, so an enabled flag will stop the rerun. Both routes display their saved records during the clean run.

The three saved replies and these two notes are the lab's comparison evidence.

## Lab steps

1. **Inspect the three versions.** Read **Week 1 Lab | Compare Prompt Design and Evidence** in Canvas. The customer complaint is the same in all three requests. Version 1 uses minimal instructions. Version 2 adds a role, six support tasks, an output format, and uncertainty instructions. Version 3 keeps Version 2 unchanged and appends findings and company rules. Before running, note what would count as a useful, supported answer.

2. **Prepare the findings and rules.** Run the notebook input cell. It reads D01, runs the diagnostic on its assigned sample, and reads the current date rule. Inspect the exact three prompt strings. Only Version 3 receives the diagnostic finding and policy text.

The finding establishes that the sample has a blank invoice date. It does not establish that the importer accepted the row or supplied a wrong date. The date rule is supplied text; you do not create a JSON policy or attach a manual. Keep expected answers out of all three requests.

3. **Run and inspect Version 1.**

Choose one route in the notebook. For browser chat, export the prompts and send Version 1 in a fresh conversation; save its full reply and capture metadata. For the CLI/API route, configure access and enable `RUN_PROMPT_PILOT` once. Inspect the actual reply and available model settings before continuing. Preserve any access error; do not invent a response.

4. **Run Versions 2 and 3.** Use the same route, model, and visible settings for Versions 2 and 3, each in a fresh conversation. Browser users save each complete reply, finish `capture.json`, and import the capture folder. CLI/API users enable `RUN_REMAINING_PROMPTS` after the pilot; each result is saved before the next request, and saved versions are reused on resume. The complete comparison uses three requests, including Version 1.

Inspect all three saved prompts and responses. Version 2 must not inherit Version 3's findings or rules. Preserve failed attempts and separate any repeated or repaired comparison. If a designated recording is required, retain its recorded status.

5. **Compare the two changes.** Write two short notes: what changed from Version 1 to Version 2, and what changed from Version 2 to Version 3? Quote the response text supporting each conclusion.

Check whether the answer distinguishes reported behavior from verified findings, identifies missing information, and follows the supplied rules. For Versions 2 and 3, inspect all six requested outputs, including the workaround, customer reply, and engineering note. A longer answer is not automatically better. An unchanged or worse response is a result to explain.

6. **Save the evidence and disclose assistance.** Keep the three selected request-response records and your two comparison notes in notebook cells, separate files, or a combination. Retain exact prompts, hashes, provenance, available model metadata, and errors. Browser users retain the exact transferred messages and source inputs. Mark unavailable metadata as unknown and identify student-reported attestations. Include your executed notebook, research note, discussion and peer response, and AI-use disclosure.

After saving the replies, set all live and browser export/import flags back to `False` before restarting and running all cells. The notebook reloads saved records. Do not replace observed failures with illustrative replies.

## Research, interaction, and disclosure

Link your primary-source research note, discussion and peer response, and AI-use disclosure. Check the three saved responses and two delta notes against the weekly rubric.

Commit your executed notebook and all required evidence to your GitHub course repository and open a Week 1 pull request. Submit the pull-request URL and evaluated commit hash in Canvas using text entry. No separate notebook upload to Canvas is required. Give the instructor repository access.